In [1]:
# imports

import numpy as np
import torch
from pathlib import Path
from ruamel.yaml import YAML
yaml = YAML()
yaml.preserve_quotes = True
import subprocess
import shutil
import subprocess, time, os, signal
from pathlib import Path

In [2]:
import trainer_and_simulator_functions as trainer

## Tests for split cost agents

### Testing theta sampler

In [4]:
theta = trainer.sample_split_cost(seed=7, n=100)
theta_prev = trainer.sample_first_thetas(100)

In [5]:
len(theta), len(theta_prev)

(100, 100)

In [8]:
tc, rc = map(float, theta[10]) 

In [9]:
tc, rc 

(0.0006666666666666666, 0.00025)

### Testing yaml-patching function

In [26]:
split_cost = True
extrinsic_reward_key: str = "extrinsic"
gamma: float = 0.99
translation_cost=theta[0][0]
turning_cost=theta[0][1]

In [27]:
template_yaml = Path("/Users/benny/Repos/octagon/Assets/Scripts/MLConfigFiles/SoloSplitCostConfig.yaml")
output_yaml = Path("/Users/benny/Documents/swc/agents/yaml/SoloSplitCostConfig.yaml")
output_yaml.parent.mkdir(parents=True, exist_ok=True) 

with template_yaml.open("r") as f:
    cfg = yaml.load(f) # loads the yaml file

if "behaviors" not in cfg or not isinstance(cfg["behaviors"], dict) or "environment_parameters" not in cfg:
    raise KeyError("Invalid yaml file format: 'behaviors' or 'environment_parameters' keys not found or 'behaviors' is not a dictionary.")
    # checks if the yaml file has the correct format

behaviours = cfg["behaviors"]
env_parameters = cfg["environment_parameters"]

# select which behaviours to patch
target_behaviours = ["OctagonAgentSolo"] if "OctagonAgentSolo" else list(behaviours.keys())
missing = [b for b in target_behaviours if b not in behaviours]
if missing:
    raise KeyError(f"Behaviours not found in yaml file: {missing}. Found: {list(behaviours.keys())}")

if not split_cost:
    if "step_penalty" not in env_parameters:
        raise KeyError("Missing environment parameter 'step_penalty'")
if split_cost:
    if "translation_cost" not in env_parameters:
        raise KeyError("Missing environment parameter 'translation_cost'")   

    if "turning_cost" not in env_parameters:
        raise KeyError("Missing environment parameter 'turning_cost'")   

for b in target_behaviours:
    bcfg = behaviours[b]

    # patch gamma and instrinsic reward strength
    reward_signals = bcfg.get("reward_signals")
    if reward_signals is None or extrinsic_reward_key not in reward_signals:
        raise KeyError(
            f"Missing reward signal '{extrinsic_reward_key}' not found in behaviour '{b}'."
            f"Available reward signals: {list(reward_signals.keys()) if isinstance(reward_signals, dict) else reward_signals}"
        )
    
    extrinsic_cfg = reward_signals[extrinsic_reward_key]

    if not isinstance(extrinsic_cfg, dict):
        raise KeyError(f"Invalid reward signal configuration format in behaviour '{b}'.")
    
    extrinsic_cfg["gamma"] = float(gamma)
    
for p in env_parameters:
    if not split_cost:
        if p == "step_penalty":
            env_parameters[p] = float(step_penalty)
    if split_cost:
        if p == "translation_cost":
            env_parameters[p] = float(translation_cost)
        if p == "turning_cost":
            env_parameters[p] = float(turning_cost)          
    
with output_yaml.open("w") as f:
    yaml.dump(cfg, f)


### Testing training launcher

In [30]:
patched_yaml = Path(output_yaml)
unity_env_path = Path("/Users/benny/Documents/swc/agents/builds/split_cost/SoloOctagon260724.app")

if not patched_yaml.exists():
    raise FileNotFoundError(f"Patched yaml file not found: {patched_yaml}")
if not unity_env_path.exists():
    raise FileNotFoundError(f"Unity environment not found: {unity_env_path}")

base_run_id: str = "split_cost_solo"
run_id = f"{base_run_id}_{0:04d}"
device: str = "cpu"
num_envs = 2
seed  = 7
cwd = Path("/Users/benny/Documents/swc/agents")

In [35]:
patched_yaml = Path(patched_yaml)
unity_env_path = Path(unity_env_path)

if not patched_yaml.exists():
    raise FileNotFoundError(f"Patched yaml file not found: {patched_yaml}")
if not unity_env_path.exists():
    raise FileNotFoundError(f"Unity environment not found: {unity_env_path}")

train_port = 5005 + 20 * 0

# build the command-line invocation
cmd = [
    "mlagents-learn", # executable
    str(patched_yaml), # path to the yaml config
    "--env", str(unity_env_path), # points to the compiled unity environment
    "--torch-device", device, # specify torch device (e.g., cuda:0)
    "--num-envs", str(num_envs), # number of parallel environments
    "--no-graphics", # headless mode
    "--run-id", run_id, # specify run id
    "--base-port", str(train_port),
    "--force",
]

if seed is not None:
    cmd.extend(["--seed", str(seed)])

# Popen allows us to monitor the output of the process in real-time, which is useful for debugging and logging.
with subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(cwd) if cwd else None
) as p:
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='')  # print each line as it is received
    p.wait()  # wait for the process to complete
    if p.returncode != 0: 
        raise subprocess.CalledProcessError(p.returncode, cmd)

/opt/miniconda3/envs/mlagents/lib/python3.10/site-packages/mlagents/torch_utils/torch.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/miniconda3/envs/mlagents/lib/python3.10/site-packages/torch/__init__.py:1144: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)

            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ 

KeyboardInterrupt: 

### Testing full simulator umbrella function with split cost

In [4]:
import os
cwd = os.getcwd()

In [23]:
cwd

'/Users/benny/Repos/agent_training'

In [9]:
in_yaml = Path("/Users/benny/Repos/octagon/Assets/Scripts/MLConfigFiles/SoloSplitCostConfig.yaml")
work_dir = cwd
unity_env_path = Path("/Users/benny/Documents/swc/agents/builds/split_cost/SoloOctagon260724.app")
step_penalty = False
split_penalty = True
behaviour_name = "OctagonAgentSolo"
n = 100
seed = 7
base_run_id = "split_cost_test_mac"


In [10]:
trainer.sbi_simulator(
    n = 100,
    in_yaml = in_yaml,
    work_dir = work_dir,
    unity_build = unity_env_path,
    base_run_id = "split_cost_test_mac",
    device = "cpu",
    simulate = True,
    n_envs = 2,
    n_eps = n,
    seed=seed,
    split_penalty=split_penalty
)

/opt/miniconda3/envs/mlagents/lib/python3.10/site-packages/mlagents/torch_utils/torch.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/miniconda3/envs/mlagents/lib/python3.10/site-packages/torch/__init__.py:1144: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)

            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ 

KeyboardInterrupt: 